In [5]:
import json
import random
import string
import nltk
import numpy as np
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Download necessary NLTK data
nltk.download("punkt")
nltk.download("wordnet")

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Sample data with intents and responses
data = {
    "intents": [
        {"tag": "greeting",
         "patterns": ["Hello", "How are you?", "Hi There", "Hi", "What's up"],
         "responses": ["Howdy Partner!", "Hello", "How are you doing?", "Greetings!", "How do you do"]
         },
        {"tag": "age",
         "patterns": ["how old are you", "when is your birthday", "when were you born"],
         "responses": ["I am 24 years old", "I was born in 1966", "My birthday is July 3rd and I was born in 1996", "03/07/1996"]
         },
        {"tag": "date",
         "patterns": ["what are you doing this weekend", "do you want to hang out sometime?", "what are your plans for this week"],
         "responses": ["I am available this week", "I don't have any plans", "I am not busy"]
         },
        {"tag": "name",
         "patterns": ["what's your name", "what are you called", "who are you"],
         "responses": ["My name is Kippi", "I'm Kippi", "Kippi"]
         },
        {"tag": "goodbye",
         "patterns": ["bye", "g2g", "see ya", "adios", "cya"],
         "responses": ["It was nice speaking to you", "See you later", "Speak Soon"]
         }
    ]
}

# Initialize words, classes, doc_x, and doc_y
words = []
classes = []
doc_x = []
doc_y = []

# Tokenize and process the data
for intent in data["intents"]:
    for pattern in intent["patterns"]:
        tokens = nltk.word_tokenize(pattern)
        words.extend(tokens)
        doc_x.append(pattern)
        doc_y.append(intent["tag"])
    if intent["tag"] not in classes:
        classes.append(intent["tag"])

# Process the words list, removing punctuation
words = [lemmatizer.lemmatize(word.lower()) for word in words if word not in string.punctuation]
words = sorted(set(words))
classes = sorted(set(classes))

# Preparing the training data
training = []
out_empty = [0] * len(classes)
for idx, doc in enumerate(doc_x):
    bow = []
    text = lemmatizer.lemmatize(doc.lower())
    for word in words:
        bow.append(1) if word in text else bow.append(0)
    output_row = list(out_empty)
    output_row[classes.index(doc_y[idx])] = 1
    training.append([bow, output_row])

random.shuffle(training)
training = np.array(training, dtype=object)

# Split the training data into input (X) and output (y)
train_X = np.array(list(training[:, 0]))
train_y = np.array(list(training[:, 1]))

# Create a Sequential model
model = Sequential()
model.add(Dense(128, input_shape=(len(train_X[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(len(train_y[0]), activation='softmax'))

# Create the Adam optimizer with a specified learning rate
adam = tf.keras.optimizers.Adam(learning_rate=0.01)

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])

print(model.summary())

# Train the model
model.fit(x=train_X, y=train_y, epochs=500, verbose=1)

# Function to clean user input text
def clean_text(text):
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

# Function to convert user input into a bag-of-words representation
def bag_of_words(text, vocab):
    tokens = clean_text(text)
    bow = [0] * len(vocab)
    for w in tokens:
        for idx, word in enumerate(vocab):
            if word == w:
                bow[idx] = 1
    return np.array(bow)

# Function to predict the user's intent based on their input
def pred_class(text, vocab, labels):
    bow = bag_of_words(text, vocab)
    result = model.predict(np.array([bow]))[0]
    thresh = 0.2
    y_pred = [[idx, res] for idx, res in enumerate(result) if res > thresh]
    y_pred.sort(key=lambda x: x[1], reverse=True)
    return_list = []
    for r in y_pred:
        return_list.append(labels[r[0]])
    return return_list

# Function to get a response from the predicted intent
def get_response(intents_list, intents_json):
    tag = intents_list[0]
    list_of_intents = intents_json["intents"]
    for i in list_of_intents:
        if i["tag"] == tag:
            result = random.choice(i["responses"])
            break
    return result

# Chatbot loop


[nltk_data] Downloading package punkt to C:\Users\maria
[nltk_data]     selciya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\maria
[nltk_data]     selciya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ (None, 128)                 │           4,864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 5)                   │             325 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 13,445 (52.52 KB)

 Trainable params: 13,445 (52.52 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 739ms/step - accuracy: 0.1053 - loss: 1.7127
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.3684 - loss: 1.5398
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.4737 - loss: 1.4429
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6316 - loss: 1.3604
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7368 - loss: 1.2123
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.8421 - loss: 1.1605
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8421 - loss: 0.9935
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.8421 - loss: 0.9029
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8947 - loss: 0.8731
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9474 - loss: 0.6744
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8947 - loss: 0.5462
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.947

In [4]:
while True:
    message = input("")
    
    # Break condition: if user types exit, quit, or bye
    if message.lower() in ["exit", "quit", "bye"]:
        print("Goodbye!")
        break

    # Use pred_class to get intents and respond
    intents = pred_class(message, words, classes)
    result = get_response(intents, data)
    print(result)


 quit


Goodbye!
